# RNA-Seq Analysis: ChrR Trisomy and the Fluconazole Response in Candida albicans

This notebook contains the RNA-seq analysis pipeline. Many
heavy steps were not actually executed inside the notebook (the bash and R cells were copied into
standalone .sh / .R files and submitted to SLURM on the ComputeCanada cluster). Cells that are meant to
run in the notebook are marked, and vice versa. The notebook exists to keep the whole workflow in one
readable narrative.

## Experimental design

18 libraries: 3 genotypes x 2 conditions x 3 biological replicates.

| Group | Sample tags | Genotype | Condition |
|-------|-------------|----------|-----------|
| A | A1, A2, A3 | WT (euploid) | YPD |
| B | B1, B2, B3 | ChrR AAB (trisomic) | YPD |
| C | C1, C2, C3 | ChrR ABB (trisomic) | YPD |
| D | D1, D2, D3 | WT (euploid) | FLZ |
| E | E1, E2, E3 | ChrR AAB (trisomic) | FLZ |
| F | F1, F2, F3 | ChrR ABB (trisomic) | FLZ |

AAB and ABB represent the two ChrR trisomy configurations (ie. which homolog of ChrR is present in two
copies). FLZ = YPD + 64ug/mL fluconazole, YPD = just YPD.

## Directory layout

Paths in this notebook assume the following structure. Python cells run from the project
root, the PCA/PERMANOVA R script runs from plotting/, and the alignment and counting scripts
run from aligning/.

```
ChrR_Seq/
├── data/                          # raw fastq.gz
├── aligning/
│   ├── star_index/
│   ├── calb_features.gtf
│   ├── calb_genome.fasta
│   ├── results/<SAMPLE>_STAR/     # STAR output, one directory per sample
│   ├── counts.txt                 # featureCounts output
│   └── edgeR_results/             # one .txt per comparison
└── plotting/
    ├── pca_PERMANOVA.R
    ├── chrR_box_plots/
    ├── volcano_plots/
        ├── ChrR_highlighted/
        └── gene_coordinates/
```

Before running anything, replace --account=your-account and --mail-user=your@email in the
SLURM headers.

Remember to load the scipy-stack/2026a module first.

In [1]:
import pandas as pd
import glob
import os, re
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from matplotlib.ticker import MaxNLocator
import matplotlib.patches as mpatches
from matplotlib.patches import Patch

## Quality Control

First, just to get a sense of how many reads there are per sample and generally how things looked, FastQC was used to check quality. Make sure you have a logs directory (mkdir -p logs) in the project root as well as in aligning/. before running anything. Copy the following into a .sh file and submit to SLURM.

Reads were not initially trimmed to avoid introducing biases into differential expression estimates (Williams et al., BMC Bioinformatics, 2016).

In [ ]:
#!/bin/bash
#SBATCH --account=your-account
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=16
#SBATCH --mem=32G
#SBATCH --time=0-4:00:00
#SBATCH --output=logs/slurm-%j.out
#SBATCH --mail-user=your@email
#SBATCH --mail-type=ALL
#SBATCH --job-name=fastqc

#Load FastQC module
module load fastqc

#Create output directory if it doesn't exist
mkdir -p fastqc_results

#Run FastQC on all fastq.gz files using 16 threads
fastqc -t 16 -o fastqc_results data/*.fastq.gz

Overrepresented sequences seem to be rRNA, which makes sense.

## Genome and Annotation

The reference genome and feature files were taken from the Candida Genome Database (CGD).

STAR requires a .gtf file to build the index, but the .gtf I found on CGD doesn't include information about whether the feature is an intron or exon or if it is rRNA. Therefore, I downloaded the current features (.gff) from CGD, moved it to .gff3, then used gffread to convert it to .gtf:

In [ ]:
conda install -c bioconda gffread

In [ ]:
gffread calb_features.gff3 -T -o calb_features.gtf

Then I built the STAR index. sjdbOverhang is set to 150 since it is the read length (151bp) minus one. --genomeSAindexNbases 11 is scaled down from the default of 14 for
this small (~14 Mb) genome, per the STAR manual's min(14, log2(genomeLength)/2 - 1) rule.

In [ ]:
STAR --runThreadN 8 \
     --runMode genomeGenerate \
     --genomeDir star_index \
     --genomeFastaFiles calb_genome.fasta \
     --sjdbGTFfile calb_features.gtf \
     --sjdbOverhang 150 \
     --genomeSAindexNbases 11

## Alignment with STAR

This is the STAR configuration I used for all samples. Save it as star_align.sh. It takes a
sample name as an argument and writes output to results/<SAMPLE>_STAR/.

--outFilterMultimapNmax 100 allows reads mapping to up to 100 loci to be
  retained, but --outSAMmultNmax 1 randomly keeps only one alignment per read to be written in the BAM (--outMultimapperOrder Random, seeded with --runRNGseed 42 for
  reproducibility). The NH tag still records the true number of loci, so you can see how many positions it mapped to if interested.

--chimSegmentMin, --chimJunctionOverhangMin, and --chimOutType were
  enabled for possible future exploration. The Chimeric.out.sam files are not used anywhere in this current analysis,  though.

--quantMode GeneCounts TranscriptomeSAM produces a ReadsPerGene.out.tab and
  Aligned.toTranscriptome.out.bam. Neither was used for the counts below (featureCounts was used
  instead), but ReadsPerGene.out.tab was helpful just as a diagnostic along the way.

In [ ]:
#!/bin/bash
#SBATCH --account=your-account
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH --time=0-48:00:00
#SBATCH --output=logs/star_%x_%j.out
#SBATCH --mail-user=your@email
#SBATCH --mail-type=ALL
#SBATCH --job-name=star_align

#Arguments
SAMPLE="$1"
STAR_INDEX="star_index"  #Path to STAR genome index directory

#Input files
IN1="../data/${SAMPLE}_R1_001.fastq.gz"
IN2="../data/${SAMPLE}_R2_001.fastq.gz"

#Output directory
OUTDIR="results/${SAMPLE}_STAR"
mkdir -p "$OUTDIR"

#Load STAR module
module load star

#Run STAR
STAR \
  --runThreadN $SLURM_CPUS_PER_TASK \
  --genomeDir $STAR_INDEX \
  --readFilesIn $IN1 $IN2 \
  --readFilesCommand zcat \
  --outFileNamePrefix "$OUTDIR/" \
  --outSAMtype BAM SortedByCoordinate \
  --outFilterIntronMotifs RemoveNoncanonical \
  --quantMode GeneCounts TranscriptomeSAM \
  --outSAMattributes NH HI AS nM XS \
  --twopassMode Basic \
  --limitBAMsortRAM 30000000000 \
  --outFilterMultimapNmax 100 \
  --winAnchorMultimapNmax 200 \
  --outSAMmultNmax 1 \
  --outMultimapperOrder Random \
  --runRNGseed 42 \
  --outFilterMatchNmin 20 \
  --outFilterMatchNminOverLread 0.15 \
  --outFilterMismatchNoverLmax 0.1 \
  --chimSegmentMin 10 \
  --chimJunctionOverhangMin 15 \
  --chimOutType SeparateSAMold \
  --outSAMunmapped Within

The following submits the script above as a separate job for each sample, which
speeds things up significantly. Run from aligning/.

In [ ]:
for fq1 in ../data/*_R1_001.fastq.gz; do
  sample=$(basename "$fq1" _R1_001.fastq.gz)
  sbatch star_align.sh "$sample"
done

Or to just run it on one sample:

In [ ]:
sbatch star_align.sh A1_S1

## Counting Reads with featureCounts

featureCounts sums the number of counts per feature. Counting was performed unstranded (-s was not specified, so featureCounts would use its default
of -s 0). This was chosen for simplicity of analysis. But the RNA-seq library prep for these samples was done with a stranded kit, so this could be modified if strandedness was of interest. Save the script below as a .sh and submit to SLURM.

-T 8 → use 8 threads

-p → paired-end data.

-B → only count fragments where both mates are successfully aligned

-C → don't count "chimera-like" pairs (mates mapping to different chromosomes / far apart),
  which reduces weird/artefactual assignments


-M --fraction → retains multi-mapping fragments (rather than discarding them) and assigns each a fractional count of 1/N, where N is read from the
NH tag. Because STAR was run with --outSAMmultNmax 1, only one alignment per multi-mapping
fragment is present in the BAM, so each such fragment contributes 1/N at a single randomly
chosen locus rather than 1/N at each of its N loci... alternatively you could keep all multi-mappers and assign a fractional count to all of them which might make counts of repetitive regions of DNA more accurate

-a calb_features.gtf → gene definitions (the same GTF used to build the STAR index)

In [ ]:
#!/bin/bash
#SBATCH --account=your-account
#SBATCH --cpus-per-task=8
#SBATCH --mem=16G
#SBATCH --time=0-4:00:00
#SBATCH --output=logs/featurecounts_%j.out
#SBATCH --job-name=featurecounts

module load subread

featureCounts -T 8 -p -B -C -M --fraction \
  -a calb_features.gtf \
  -o counts.txt \
  results/*_STAR/Aligned.sortedByCoord.out.bam

## Differential Expression (edgeR)

Save the R script below as edgeR.R and run it using the SLURM script that follows.

Low-expression genes are filtered with filterByExpr prior to fitting. Two interaction
contrasts are extracted (AAB×FLZ, ABB×FLZ), and all pairwise group comparisons are generated in both directions.

In [ ]:
library(edgeR)

#Read counts.txt (from featureCounts)
counts <- read.delim("counts.txt", comment.char="#", check.names=FALSE) #ignore lines starting with "#" (header)

#Create a count matrix from the sorted BAM files (one per sample)
bam_cols <- grep("Aligned.sortedByCoord.out.bam$", colnames(counts))
count_matrix <- counts[, bam_cols]
rownames(count_matrix) <- counts$Geneid #row names are from the GeneID column

#Define the sample groupings: 6 groups, 3 replicates each
#NOTE: this assignment is POSITIONAL and assumes counts.txt columns are in the order
#A1,A2,A3,B1,...,F3. Print colnames below and confirm before trusting any downstream result.
group <- factor(c(rep("A",3), rep("B",3), rep("C",3), rep("D",3), rep("E",3), rep("F",3)))

#Confirm count matrix columns match expected number, and show the order for manual checking
stopifnot(ncol(count_matrix) == length(group))
print(data.frame(column = colnames(count_matrix), assigned_group = group))

#Map A-F to the specific genotype/condition
group_to_genotype <- c(
  "A"="WT",   "B"="TriA", "C"="TriB",
  "D"="WT",   "E"="TriA", "F"="TriB"
)
group_to_treatment <- c(
  "A"="YPD",  "B"="YPD",  "C"="YPD",
  "D"="FLZ",  "E"="FLZ",  "F"="FLZ"
)

#Build a sample information table
sampleTable <- data.frame(
  sample    = colnames(count_matrix),
  group     = group,
  #Looks up the genotype corresponding to each A-F group
  genotype  = factor(group_to_genotype[as.character(group)],  levels=c("WT","TriA","TriB")),
  #Same thing for treatment
  treatment = factor(group_to_treatment[as.character(group)], levels=c("YPD","FLZ")),
  stringsAsFactors = FALSE
)

#Create an edgeR DGEList object (takes the plain count matrix created above and turns it into an edgeR object)
dge <- DGEList(counts=count_matrix)
#Corrects for differences in number of reads between libraries, etc.
dge <- calcNormFactors(dge)

#Filter low-expression genes. This is important because testing for differential expression for
#many genes simultaneously adds to the multiple testing burden, reducing the power to detect DE
#genes. With edgeR, filterByExpr automatically sets the CPM threshold (k) as what would
#correspond to ~10 raw counts, which is standard.
keep <- filterByExpr(dge, group=interaction(sampleTable$genotype, sampleTable$treatment))
dge <- dge[keep, , keep.lib.sizes=FALSE]
cat("Genes retained after filterByExpr:", sum(keep), "of", length(keep), "\n")

#Design statistical model based on the interaction: genotype x treatment
design <- model.matrix(~ genotype * treatment, data=sampleTable)
print(colnames(design))

#Fit model (quasi-likelihood)
#This basically estimates dispersion (how much variation there is between replicates)
dge <- estimateDisp(dge, design)
#For every gene, estimate how much expression is associated with genotype, FLZ, and their interaction, while accounting for variability
fit <- glmQLFit(dge, design)

#Model QC
#Rough summary of biological variability
cat("Common dispersion:", dge$common.dispersion, "\n")
cat("BCV (sqrt common dispersion):", sqrt(dge$common.dispersion), "\n")

#Make output directory if needed
if (!dir.exists("edgeR_results")) dir.create("edgeR_results")

#Helper to write results in a format the Python cells expect:
#rownames = genes; includes logFC and FDR columns.
write_edger <- function(test_obj, out_file) {
  tab <- topTags(test_obj, n=Inf)$table
  #Keep standard edgeR columns (includes logFC, FDR)
  write.table(tab, file=out_file, sep="\t", quote=FALSE, col.names=NA)
  message("Wrote: ", out_file)
}

#Interaction tests. logFC here is: (FLZ - YPD) in trisomy minus (FLZ - YPD) in WT
#Trisomy ChrR_AAB x FLZ interaction: (TriA_FLZ - TriA_YPD) - (WT_FLZ - WT_YPD)
test_TriA_int <- glmQLFTest(fit, coef="genotypeTriA:treatmentFLZ")
write_edger(test_TriA_int, file.path("edgeR_results", "edger_TriA_interaction.txt"))

#Trisomy ChrR_ABB x FLZ interaction: (TriB_FLZ - TriB_YPD) - (WT_FLZ - WT_YPD)
test_TriB_int <- glmQLFTest(fit, coef="genotypeTriB:treatmentFLZ")
write_edger(test_TriB_int, file.path("edgeR_results", "edger_TriB_interaction.txt"))

#Make another model for all other simple pairwise comparisons too.
#These are fitted from a SEPARATE model (group-only design, dispersions re-estimated),
#So the logFC values here are not numerically interchangeable with the interaction results above.
group2 <- factor(sampleTable$group, levels=c("A","B","C","D","E","F"))
design_group <- model.matrix(~0 + group2)
colnames(design_group) <- levels(group2)

#Estimate dispersion and fit GLM
#Make a new edgeR object using the already-filtered counts from dge above
dge2 <- DGEList(counts=dge$counts)
#Corrects for differences in number of reads between libraries, etc. again
dge2 <- calcNormFactors(dge2)
#Estimate dispersion for this second model
dge2 <- estimateDisp(dge2, design_group)
#Again, for every gene fit expression across the six experimental groups while accounting for variability
fit2 <- glmQLFit(dge2, design_group)

#Generate all pairwise comparisons in both directions (A_vs_B and B_vs_A, etc.)
group_names <- colnames(design_group)
for (i in seq_along(group_names)) {
  for (j in seq_along(group_names)) {
    if (i != j) {
      g1 <- group_names[i]
      g2 <- group_names[j]
      #Build contrast vector for g1 vs g2 (g1-g2)
      contrast <- rep(0, length(group_names))
      contrast[i] <- 1
      contrast[j] <- -1
      names(contrast) <- group_names
      test_pair <- glmQLFTest(fit2, contrast=contrast)
      #Output result with filename indicating direction
      out_file <- file.path("edgeR_results", paste0("edger_", g1, "_vs_", g2, ".txt"))
      write_edger(test_pair, out_file)
    }
  }
}

Submit with the following:

In [ ]:
#!/bin/bash
#SBATCH --account=your-account
#SBATCH --cpus-per-task=8
#SBATCH --mem=16G
#SBATCH --time=0-0:10:00
#SBATCH --output=logs/edgeR_%j.out
#SBATCH --mail-user=your@email
#SBATCH --mail-type=ALL
#SBATCH --job-name=edgeR

module load StdEnv/2023
module load gcc/12.3
module load r-bundle-bioconductor/3.20

Rscript edgeR.R

The edgeR script writes one .txt file per comparison. The following cell merges them into a
single .csv for convenience. Run this within the notebook:

In [4]:
input_dir = "aligning/edgeR_results"
output_file = "aligning/edgeR_results/combined_edgeR_results.csv"

files = sorted(glob.glob(os.path.join(input_dir, "*.txt")))
print(f"Found {len(files)} files")

dfs = []
for f in files:
    comp = os.path.splitext(os.path.basename(f))[0]  #Filename without extension

    #Read with gene names as the index
    df = pd.read_csv(f, sep="\t", header=0, index_col=0)

    #Prefix comparison name to each column
    df = df.add_prefix(f"{comp}_")
    dfs.append(df)

#Merge everything on gene index
combined = pd.concat(dfs, axis=1)
combined.to_csv(output_file)
print("Saved:", output_file, combined.shape)

Found 32 files
Saved: aligning/edgeR_results/combined_edgeR_results.csv (12726, 160)


## PCA and PERMANOVA

Save the R script below as pca_PERMANOVA.R in ChrR_Seq/plotting/ and run it with
Rscript pca_PERMANOVA.R. No job submission is needed since it is quick. The script operates
on TMM-normalised, filtered logCPM values and performs:

PCA, PERMANOVA (with adonis2): how much variance is explained by drug, genotype, and their interaction. Dispersion checks (with betadisper): confirms PERMANOVA results are not confounded by unequal group spread.

You may have to run this (bash) in your terminal before running the R script:

In [ ]:
module load r/4.3.1
module load r-bundle-bioconductor/3.21
module load edgeR/4.6.3

Some of the startup code here is redundant with the DE (featureCounts) code above, so this could technically be modified if everything is always being run start-to-finish:

In [ ]:
suppressPackageStartupMessages({
  library(edgeR)
  library(ggplot2)
  library(vegan)     #adonis2, betadisper
})

#Set the random-number seed
set.seed(1)
#Set to 999 for number of permutations for the tests
perm <- 999

#Load featureCounts results
counts <- read.delim("../aligning/counts.txt", comment.char="#", check.names = FALSE) #Ignore lines with "#" (header)
#Create a count matrix from the sorted BAM files (one per sample)
bam_cols <- grep("Aligned.sortedByCoord.out.bam$", colnames(counts))
count_matrix <- counts[, bam_cols, drop = FALSE]
rownames(count_matrix) <- counts$Geneid

#Define sample groups (matches column order in counts.txt)
group <- factor(c(
  rep("WT YPD", 3),
  rep("ChrR AAB YPD", 3),
  rep("ChrR ABB YPD", 3),
  rep("WT FLZ", 3),
  rep("ChrR AAB FLZ", 3),
  rep("ChrR ABB FLZ", 3)
))

#Extract treatment information
Drug <- factor(ifelse(grepl("FLZ", group), "FLZ", "YPD"), levels = c("YPD","FLZ"))
#Extract genotype information
Genotype <- gsub(" (YPD|FLZ)$", "", as.character(group))
Genotype <- factor(Genotype, levels = c("WT", "ChrR AAB", "ChrR ABB"))
#Simplify genotype into just WT vs trisomy (useful for later when looking at effects of trisomy, generally)
TrisomyStatus <- factor(ifelse(Genotype == "WT", "WT", "Trisomy"),
                        levels = c("WT","Trisomy"))
#Safety checks
stopifnot(length(group) == ncol(count_matrix))
stopifnot(all(!is.na(Drug)))
stopifnot(all(!is.na(Genotype)))

#Print the dataframe so you can verify that the samples are labeled properly
print(data.frame(column = colnames(count_matrix), assigned_group = group))

#Create a metadata table for the PERMANOVA later
meta <- data.frame(
  Sample = colnames(count_matrix),
  Group = group,
  Drug = Drug,
  Genotype = Genotype,
  TrisomyStatus = TrisomyStatus,
  stringsAsFactors = FALSE
)

#Normalize + filter lowly expressed genes + logCPM
dge <- DGEList(counts = count_matrix, group = group) #Create an edgeR DGEList object (takes the plain count matrix created above and turns it into an edgeR object)
dge <- calcNormFactors(dge) #Corrects for differences in number of reads between libraries, etc.
#Filter low-expression genes. This is important because testing for differential expression for
#many genes simultaneously adds to the multiple testing burden, reducing the power to detect DE
#genes. With edgeR, filterByExpr automatically sets the CPM threshold (k) as what would
#correspond to ~10 raw counts, which is standard.
keep <- filterByExpr(dge, group = group)
dge <- dge[keep, , keep.lib.sizes = FALSE]
logCPM <- cpm(dge, log = TRUE, prior.count = 1)     #Genes x Samples
logCPM_t <- t(logCPM)                               #Samples x Genes

###PCA###
#Genes are centred but NOT scaled
pca <- prcomp(logCPM_t, center = TRUE, scale. = FALSE) #Each gene has its mean expression subtracted before PCA, and we do not force every gene to have the same variance
var_exp <- (pca$sdev^2) / sum(pca$sdev^2) #Calculates the fraction of total variation explained by each PC
pc1_lab <- paste0("PC1 (", round(var_exp[1] * 100, 1), "%)") #x-axis label
pc2_lab <- paste0("PC2 (", round(var_exp[2] * 100, 1), "%)") #y-axis label
#Create the PCA data table (contains sample info and PCA coordinates for each)
pc_data <- data.frame(
  PC1 = pca$x[, 1],
  PC2 = pca$x[, 2],
  Group = group,
  stringsAsFactors = FALSE
)

#PCA plot
p <- ggplot(pc_data, aes(x = PC1, y = PC2, color = Group)) +
  geom_point(size = 3) + #sample datapoint size
  coord_fixed() +
  labs( #plot labels
    title = "PCA of RNA-seq Samples",
    x = pc1_lab,
    y = pc2_lab
  ) +
  theme_classic(base_family = "Arial") +
  theme(
    plot.title   = element_text(size = 12, face = "bold", hjust = 0.5),
    axis.title.x = element_text(size = 12, face = "bold"),
    axis.title.y = element_text(size = 12, face = "bold"),
    axis.text.x  = element_text(size = 12, face = "bold"),
    axis.text.y  = element_text(size = 12, face = "bold"),
    legend.title = element_text(size = 12, face = "bold"),
    legend.text  = element_text(size = 12, face = "bold"),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1)
  )
ggsave("pca_plot.png", plot = p, width = 5, height = 3, dpi = 300)


###Distances for PERMANOVA (same space as the PCA above)... ie. calculates the Euclidean distance between every pair of samples using all retained genes
dist_mat <- dist(logCPM_t, method = "euclidean")
#Dispersion checks
bd_drug  <- betadisper(dist_mat, meta$Drug) #Tests how spread out are samples are within the two treatment groups
bd_geno  <- betadisper(dist_mat, meta$Genotype) #Tests how spread out are samples are within the three genotypes
bd_group <- betadisper(dist_mat, meta$Group)  #Tests spread among all 6 treatment+genotype groups
bd_drug_anova  <- anova(bd_drug) #Tests whether the treatment groups have significantly different dispersion
bd_geno_anova  <- anova(bd_geno) #Tests whether the genotype groups have significantly different dispersion
bd_group_anova <- anova(bd_group) #Tests whether the full treatment+genotype groups have significantly different dispersion

###Main PERMANOVA###
#Across the whole transcriptome, how much variation is associated with drug and how much is associated with genotype? 
#Margin means each variable is tested while accounting for the other (no interaction)
adon_main_margin <- adonis2(
  dist_mat ~ Drug + Genotype,
  data = meta,
  permutations = perm,
  by = "margin"
)
#This now checks for how much variation drug + genotype explain together
adon_main_total <- adonis2(
  dist_mat ~ Drug + Genotype,
  data = meta,
  permutations = perm,
  by = NULL
)
#And this then tests for whether there is a drug x genotype interaction (explicit, marginal)
adon_int_margin <- adonis2(
  dist_mat ~ Drug * Genotype,
  data = meta,
  permutations = perm,
  by = "margin"
)
#This is the same thing but tests for drug x trisomy (generally) interaction (lumps AAB and ABB together)
adon_int_trisomy_margin <- adonis2(
  dist_mat ~ Drug * TrisomyStatus,
  data = meta,
  permutations = perm,
  by = "margin"
)

#Summary
cat("\nPCA + PERMANOVA summary\n")
cat("\nDispersion checks:\n")
cat("Drug:     P =", bd_drug_anova$`Pr(>F)`[1], "\n")
cat("Genotype: P =", bd_geno_anova$`Pr(>F)`[1], "\n")
cat("6 groups: P =", bd_group_anova$`Pr(>F)`[1], "\n")
cat("\nMain PERMANOVA effects:\n")
cat(
  "Drug:     R2 =", round(adon_main_margin["Drug", "R2"], 3),
  " P =", adon_main_margin["Drug", "Pr(>F)"], "\n"
)
cat(
  "Genotype: R2 =", round(adon_main_margin["Genotype", "R2"], 3),
  " P =", adon_main_margin["Genotype", "Pr(>F)"], "\n"
)
cat(
  "Drug + Genotype total: R2 =",
  round(adon_main_total["Model", "R2"], 3), "\n"
)
cat("\nInteractions:\n")
cat(
  "Drug x Genotype: R2 =",
  round(adon_int_margin["Drug:Genotype", "R2"], 3),
  " P =", adon_int_margin["Drug:Genotype", "Pr(>F)"], "\n"
)
cat(
  "Drug x TrisomyStatus: R2 =",
  round(adon_int_trisomy_margin["Drug:TrisomyStatus", "R2"], 3),
  " P =", adon_int_trisomy_margin["Drug:TrisomyStatus", "Pr(>F)"], "\n"
)

## ChrR vs. Non-ChrR Expression Boxplots

The following cell generates 4-panel boxplots comparing the normalised expression of ChrR genes vs. all other genes in each trisomic strain relative to WT, under both YPD and FLZ conditions. Expression ratios are centered to the non-ChrR median, and a Mann-Whitney U test is computed for each panel. Run within the notebook.

In [6]:
#Paths
COUNTS_PATH = "aligning/counts.txt"
OUTDIR = "plotting/chrR_box_plots"
os.makedirs(OUTDIR, exist_ok=True) #Makes the output folder if it doesn't exist
BAM_COL_REGEX = r"Aligned\.sortedByCoord\.out\.bam$"
PSEUDOCOUNT_CPM = 0.1 #Add 0.1 CPM to both numerator and denominator when calculating ratios (avoids divide by 0 issues)

#Filtering: keep genes with CPM >= ~10 counts (median library) in at least 3 samples
MIN_SAMPLES_WITH_SIGNAL = 3

#Plot formatting
FIGSIZE = (8.5, 5)
TRISOMIC_COLOR = "red"
LINEWIDTH = 1
SHOW_FLIERS = False #Hides boxplot outlier points (does not remove them from statistical tests, just from plot to avoid poor readability)
CENTER_TO_EUPLOID_MEDIAN = True  #Linear ratios centered to non-ChrR median
YLIM_PAD_FRAC = 0.10 #Shared y-axis padding

#Sample column tags
GROUPS = {
    ("YPD", "WT"):  ["A1_", "A2_", "A3_"],
    ("YPD", "AAB"): ["B1_", "B2_", "B3_"],
    ("YPD", "ABB"): ["C1_", "C2_", "C3_"],
    ("FLZ", "WT"):  ["D1_", "D2_", "D3_"],
    ("FLZ", "AAB"): ["E1_", "E2_", "E3_"],
    ("FLZ", "ABB"): ["F1_", "F2_", "F3_"],
}
#Which comparisons to plot
PANELS = [
    ("YPD", "AAB", "YPD: ChrR AAB vs WT"),
    ("YPD", "ABB", "YPD: ChrR ABB vs WT"),
    ("FLZ", "AAB", "FLZ: ChrR AAB vs WT"),
    ("FLZ", "ABB", "FLZ: ChrR ABB vs WT"),
]

#Helpers

#For every column name, keeps it if it contains any of the provided sample group tags
def select_cols(cols, substrings):
    return [c for c in cols if any(s in c for s in substrings)]
    
#Identify ChrR genes
def is_chrr(gene_id: str) -> bool: 
    return str(gene_id).upper().startswith("CR") #Converts the gene ID to uppercase and checks whether it begins with "CR"

#Calculate CPM by summing all gene counts, and diving each gene count by the total (and multiplying by 1,000,000)
def cpm(counts_df: pd.DataFrame) -> pd.DataFrame:
    lib = counts_df.sum(axis=0).replace(0, np.nan)
    return counts_df.div(lib, axis=1) * 1e6

#Convert p-values to stars
def stars(p):
    if p < 1e-4: return "****"
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 0.05: return "*"
    return "ns"

#Calculate expression ratios
def ratio_panel(mean_cpm, chrR_mask, cond, tri_geno):
    tri = mean_cpm[(cond, tri_geno)] #Gets the mean CPM for the trisomic strain in that condition
    wt  = mean_cpm[(cond, "WT")] #Gets the matching WT mean CPM (matched condition)
    r = (tri + PSEUDOCOUNT_CPM) / (wt + PSEUDOCOUNT_CPM) #For every gene, calculated mean trisomy expression vs. mean WT expression (plus the pseudocounts)
    if CENTER_TO_EUPLOID_MEDIAN:
        center = float(np.median(r[~chrR_mask].values)) #Finds the median trisomy/WT ratio among non-ChrR genes
        if center > 0:
            r = r / center #Divide every gene's ratio by that median
    return r[~chrR_mask].values, r[chrR_mask].values, r

#Calculate boxplot whiskers
def whisker_bounds_iqr(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan, np.nan
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    x_sorted = np.sort(x)
    lo_clip = x_sorted[x_sorted >= lo][0] if np.any(x_sorted >= lo) else x_sorted[0]
    hi_clip = x_sorted[x_sorted <= hi][-1] if np.any(x_sorted <= hi) else x_sorted[-1]
    return lo_clip, hi_clip

#Draw each boxplot
def box_panel(ax, eu, tri, title, ylims, show_ylabel=False):
    eu = np.asarray(eu, float); eu = eu[np.isfinite(eu)]
    tri = np.asarray(tri, float); tri = tri[np.isfinite(tri)]
    bp = ax.boxplot(
        [eu, tri],
        positions=[1, 2],
        widths=0.55,
        patch_artist=True,
        showfliers=SHOW_FLIERS,
        boxprops=dict(linewidth=LINEWIDTH, color="black"),
        whiskerprops=dict(linewidth=LINEWIDTH, color="black"),
        capprops=dict(linewidth=LINEWIDTH, color="black"),
        medianprops=dict(linewidth=LINEWIDTH, color="black"),
    )
    bp["boxes"][0].set_facecolor("white")
    bp["boxes"][1].set_facecolor(TRISOMIC_COLOR)
    bp["boxes"][1].set_alpha(0.85)
    ax.set_ylim(*ylims)
    span = ylims[1] - ylims[0]
    #Significance bracket
    p = mannwhitneyu(eu, tri, alternative="two-sided").pvalue
    sig = stars(p)
    #Report the results alongside the test, for the figure legend
    print(f"  {title}: median non-ChrR = {np.median(eu):.3f}, median ChrR = {np.median(tri):.3f}, "
          f"n_nonChrR = {eu.size}, n_ChrR = {tri.size}, Mann-Whitney p = {p:.3g}")
    #Place bracket above the upper whisker for each group
    _, eu_hi = whisker_bounds_iqr(eu)
    _, tri_hi = whisker_bounds_iqr(tri)
    top_whisker = np.nanmax([eu_hi, tri_hi])
    y = top_whisker + 0.015 * span
    h = 0.03 * span
    ax.plot([1, 1, 2, 2], [y, y + h, y + h, y], lw=LINEWIDTH, color="black")
    ax.text(1.5, y + h * 1.1, sig, ha="center", va="bottom",
            fontsize=10, fontweight="bold")
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Non-ChrR", "ChrR"], fontweight="bold", fontsize=10)
    ax.set_title(title, fontweight="bold", fontsize=10)
    ax.yaxis.set_major_locator(MaxNLocator(5))
    ax.tick_params(axis="y", labelsize=10)
    for t in ax.get_yticklabels():
        t.set_fontweight("bold")
    if show_ylabel:
        ax.set_ylabel(
            "Normalized Expression\n(Centered to Non-ChrR Median)",
            fontweight="bold", fontsize=10
        )
    else:
        ax.set_ylabel("")
    for spine in ax.spines.values():
        spine.set_linewidth(LINEWIDTH)

#Main
counts = pd.read_csv(COUNTS_PATH, sep="\t", comment="#") #Read  counts.txt
#Selects the genomic coordinate-sorted BAM columns and make a matrix with gene ID rows and sample count columns
bam_cols = [c for c in counts.columns if re.search(BAM_COL_REGEX, c)]
counts_mat = counts.set_index("Geneid")[bam_cols]

#Run the CPM function defined above
cpm_mat = cpm(counts_mat)

#Calculate the CPM corresponding to approximately 10 raw reads in the median-sized library
median_lib_m = float(np.median(counts_mat.sum(axis=0))) / 1e6
cpm_thresh = 10.0 / median_lib_m
#Keep genes meeting the CPM threshold in at least 3 samples. Basically recreates filterByExpr from edgeR in that analysis
keep = (cpm_mat >= cpm_thresh).sum(axis=1) >= MIN_SAMPLES_WITH_SIGNAL
cpm_mat = cpm_mat.loc[keep]

#Build per-group mean CPM
group_cols = {(cond, geno): select_cols(bam_cols, subs) for (cond, geno), subs in GROUPS.items()}
#Checks that exactly 3 BAM files are found for every group (safety check)
for key, cols in group_cols.items():
    assert len(cols) == 3, f"Group {key} matched {len(cols)} columns, expected 3: {cols}"
mean_cpm = {(cond, geno): cpm_mat[cols].mean(axis=1) for (cond, geno), cols in group_cols.items()} #For every gene and every group, average the three replicate CPM values
#Gets the gene IDs and checks to see if it is a ChrR gene
gene_ids = mean_cpm[("FLZ", "WT")].index.astype(str)
chrR_mask = np.array([is_chrr(g) for g in gene_ids])
#Just prints how many genes passed filtering and how many are ChrR vs. non-ChrR
print(f"Genes retained: {len(gene_ids)}  (ChrR: {chrR_mask.sum()}, non-ChrR: {(~chrR_mask).sum()})")

#Compute all panels first
panel_data = []
all_whisker_lows = []
all_whisker_highs = []
#Loops through the four planned comparisons
for cond, tri, title in PANELS:
    eu, chrr, r = ratio_panel(mean_cpm, chrR_mask, cond, tri) #Calculates the trisomy/WT ratios for the given panel
    panel_data.append((eu, chrr, title))

    #Store the lowest and highest whiskers across all panels so we can have one shared y-axis scale later for the plot
    eu_lo, eu_hi = whisker_bounds_iqr(eu)
    ch_lo, ch_hi = whisker_bounds_iqr(chrr)
    all_whisker_lows.append(np.nanmin([eu_lo, ch_lo]))
    all_whisker_highs.append(np.nanmax([eu_hi, ch_hi]))

#Finds the lowest and highest whisker anywhere in the four-panel figure and adds 10% padding
lo = float(np.nanmin(all_whisker_lows))
hi = float(np.nanmax(all_whisker_highs))
pad = YLIM_PAD_FRAC * (hi - lo if hi > lo else 1.0)
ylims = (max(0, lo - pad), hi + pad)

#Make the four-panel figure
plt.rcParams["font.family"] = "DejaVu Sans"
fig, axes = plt.subplots(1, 4, figsize=FIGSIZE, sharey=True) #sharey=True means all panels use the same y-axis

for ax, (eu, chrr, title) in zip(axes, panel_data):
    box_panel(ax, eu, chrr, title, ylims, show_ylabel=(ax is axes[0]))

plt.tight_layout()
out_png = os.path.join(OUTDIR, "ChrR_boxplots_YPD_and_FLZ_4panel.png")
plt.savefig(out_png, dpi=300) #Save the figure
plt.close()
print("Saved:", out_png)

Genes retained: 12727  (ChrR: 2076, non-ChrR: 10651)
  YPD: ChrR AAB vs WT: median non-ChrR = 1.000, median ChrR = 1.356, n_nonChrR = 10651, n_ChrR = 2076, Mann-Whitney p = 0
  YPD: ChrR ABB vs WT: median non-ChrR = 1.000, median ChrR = 1.381, n_nonChrR = 10651, n_ChrR = 2076, Mann-Whitney p = 0
  FLZ: ChrR AAB vs WT: median non-ChrR = 1.000, median ChrR = 1.384, n_nonChrR = 10651, n_ChrR = 2076, Mann-Whitney p = 0
  FLZ: ChrR ABB vs WT: median non-ChrR = 1.000, median ChrR = 1.403, n_nonChrR = 10651, n_ChrR = 2076, Mann-Whitney p = 0
Saved: plotting/chrR_box_plots/ChrR_boxplots_YPD_and_FLZ_4panel.png


## Volcano Plots

Generates volcano plots for all of the comparisons listed in desired_comparisons, with ChrR genes highlighted in black and non-ChrR significant genes in red. Run within the notebook.

In [11]:
#Paths
results_dir = "aligning/edgeR_results/"
chrr_dir = "plotting/volcano_plots/ChrR_highlighted/"
coords_dir = "plotting/volcano_plots/gene_coordinates"
os.makedirs(chrr_dir, exist_ok=True)
os.makedirs(coords_dir, exist_ok=True)

#Settings
fdr_thresh = 0.05
logfc_thresh = 0 #We don't really need a logFC threshold since we are still interested in subtle (significant) changes, not just huge changes in a select set of genes
POINT_SIZE = 13

#Which sample comparisons do we want to plot
desired_comparisons = [
    "B_vs_A", "C_vs_A", "C_vs_B", "D_vs_A", "E_vs_B", "F_vs_C", "E_vs_D", "F_vs_D", "F_vs_E",
    "TriA_interaction", "TriB_interaction",
]

#In-panel description of each contrast
#Remember that A = WT YPD, B = AAB YPD, C = ABB YPD, D = WT FLZ, E = AAB FLZ, F = ABB FLZ
SHORT_LABELS = {
    "B_vs_A": "ChrR AAB vs WT\n(YPD)",
    "C_vs_A": "ChrR ABB vs WT\n(YPD)",
    "C_vs_B": "ChrR ABB vs ChrR AAB\n(YPD)",
    "D_vs_A": "WT\n(FLZ vs YPD)",
    "E_vs_B": "ChrR AAB\n(FLZ vs YPD)",
    "F_vs_C": "ChrR ABB\n(FLZ vs YPD)",
    "E_vs_D": "ChrR AAB vs WT\n(FLZ)",
    "F_vs_D": "ChrR ABB vs WT\n(FLZ)",
    "F_vs_E": "ChrR ABB vs ChrR AAB\n(FLZ)",
    "TriA_interaction": "ChrR AAB × FLZ\nInteraction",
    "TriB_interaction": "ChrR ABB × FLZ\nInteraction",
}

#Labels
def plot_labels_for_comp(comp: str):
    if comp == "TriA_interaction":
        return ("ChrR AAB × FLZ Interaction (ΔΔlogFC)",
                "ΔΔlog2FC = [(ChrR AAB FLZ - ChrR AAB YPD)]\n− [(WT FLZ - WT YPD)]")
    if comp == "TriB_interaction":
        return ("ChrR ABB × FLZ Interaction (ΔΔlogFC)",
                "ΔΔlog2FC = [(ChrR ABB FLZ - ChrR ABB YPD)]\n− [(WT FLZ - WT YPD)]")
    return (comp.replace("_", " "), "log2 Fold Change")

#This loops through each comparison, loads the corresponding edgeR file, annotates significance and ChrR status, then saves the volcano coordinates and plots
for i, comp in enumerate(desired_comparisons, start=1): #Goes through every entry in desired_comparisons
    df = pd.read_csv(os.path.join(results_dir, f"edger_{comp}.txt"), sep="\t", index_col=0) #Load the edgeR results
    df["is_ChrR"] = df.index.str.upper().str.startswith("CR") #Mark which genes are on ChrR
    df["minus_log10_FDR"] = -np.log10(df["FDR"].replace(0, np.nextafter(0, 1))) #Do the -log10 transformation of the adjusted p-values (FDR)
    df["significant"] = (df["FDR"] < fdr_thresh) & (df["logFC"].abs() > logfc_thresh) #Define significant genes based on FDR and logFC (if any) thresholds

    #Significant up / down counts
    n_up = int((df["significant"] & (df["logFC"] > 0)).sum()) #Count significantly upregulated genes
    n_down = int((df["significant"] & (df["logFC"] < 0)).sum()) #Count significantly downregulated genes

    #Export coordinates + significance
    coords = df[["logFC", "minus_log10_FDR", "FDR", "significant", "is_ChrR"]].copy()
    coords.index.name = "Gene"
    coords.to_csv(os.path.join(coords_dir, f"volcano_coords_{comp}.csv"))
    title, xlab = plot_labels_for_comp(comp)

    #ChrR-highlighted volcano colors
    colors = np.where(df["significant"] & df["is_ChrR"], "black",
             np.where(df["significant"], "red", "grey"))
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(df["logFC"], df["minus_log10_FDR"], c=colors, s=POINT_SIZE, edgecolors="none") #Generate all gene datapoints
    ax.set_box_aspect(1)
    ax.margins(x=0.2, y=0.2)
    ax.axhline(-np.log10(fdr_thresh), color="black", ls="--", lw=1)
    ax.set_xlabel(xlab, fontsize=11, fontweight="bold")
    ax.set_ylabel("-log10 FDR", fontsize=11, fontweight="bold")
    ax.tick_params(axis="both", labelsize=10)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")

    #In-panel annotations (axes coords, so they stay pinned to the corners)
    ax.text(0.02, 0.98, SHORT_LABELS.get(comp, comp.replace("_", " ")),
            transform=ax.transAxes, ha="left", va="top",
            fontsize=9, fontweight="bold", linespacing=1.3)
    ax.text(0.98, 0.98, f"↑ n = {n_up}\n↓ n = {n_down}",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=9, fontweight="bold", linespacing=1.3)
    #Build the legend. Grey isn't included because "not significant" datapoints should be fairly obvious based on the threshold, but could be added here
    leg = ax.legend(
        handles=[
            mpatches.Patch(color="black", label="Significant ChrR Genes"),
            mpatches.Patch(color="red", label="Non-ChrR Significant Genes"),
        ],
        loc="lower center",
        bbox_to_anchor=(0.5, 1.0),
        frameon=False,
        fontsize=10,
        ncol=1
    )
    for text in leg.get_texts():
        text.set_fontweight("bold") #Makes legend text bold
    
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    out_png = os.path.join(chrr_dir, f"volcano_{comp}_ChrR.png")
    plt.savefig(out_png, dpi=300, bbox_inches="tight", pad_inches=0.2) #Save the given plot
    plt.close()
    print(f"Saved: {out_png}  (up={n_up}, down={n_down})")

Saved: plotting/volcano_plots/ChrR_highlighted/volcano_B_vs_A_ChrR.png  (up=2489, down=2661)
Saved: plotting/volcano_plots/ChrR_highlighted/volcano_C_vs_A_ChrR.png  (up=2979, down=2926)
Saved: plotting/volcano_plots/ChrR_highlighted/volcano_C_vs_B_ChrR.png  (up=1484, down=1036)
Saved: plotting/volcano_plots/ChrR_highlighted/volcano_D_vs_A_ChrR.png  (up=5282, down=4807)
Saved: plotting/volcano_plots/ChrR_highlighted/volcano_E_vs_B_ChrR.png  (up=5060, down=4939)
Saved: plotting/volcano_plots/ChrR_highlighted/volcano_F_vs_C_ChrR.png  (up=4909, down=4838)
Saved: plotting/volcano_plots/ChrR_highlighted/volcano_E_vs_D_ChrR.png  (up=2551, down=3467)
Saved: plotting/volcano_plots/ChrR_highlighted/volcano_F_vs_D_ChrR.png  (up=2816, down=3780)
Saved: plotting/volcano_plots/ChrR_highlighted/volcano_F_vs_E_ChrR.png  (up=1027, down=830)
Saved: plotting/volcano_plots/ChrR_highlighted/volcano_TriA_interaction_ChrR.png  (up=1202, down=1212)
Saved: plotting/volcano_plots/ChrR_highlighted/volcano_TriB_i

## Significant DEGs per Chromosome

Reads all of the comparison coordinate CSVs generated by the volcano plot cell above and then sums the number of up- and down-regulated significant genes per chromosome for each comparison. Outputs both a long-format and wide-format CSV. Run within the notebook.

The long table also reports n_tested and frac_sig per chromosome. Keep in mind that the raw DEG counts are not comparable across chromosomes from this raw output (due to differences in number of genes per chromosome), hence frac_sig per chromosome. Counts were processed further in Excel (see Supplementary Tables in paper) anyways.

In [9]:
#Paths
coords_dir = "plotting/volcano_plots/gene_coordinates"
out_csv = os.path.join(coords_dir, "sig_DEGs_per_chromosome_all_comparisons.csv")

#Function that converts gene IDs into the corresponding chromosome (CR and CM are explicitly listed since R and M are characters), the rest can be defined by a number
def gene_to_chr(gene_id: str) -> str:
    g = str(gene_id).upper()
    if g.startswith("CR"):
        return "ChrR"
    if g.startswith("CM"):
        return "ChrM"
    m = re.match(r"^C(\d+)", g)
    if m:
        return f"Chr{m.group(1)}"
    return "Other/Unknown"

#Create an empty list to store the results eventually
rows = []
#Gets every filename in coords_dir, sorts them alphabetically, and loops through them one at a time
for fn in sorted(os.listdir(coords_dir)):
    if not fn.endswith(".csv"): #Skip if the file is not a .csv for some reason
        continue
    if not fn.startswith("volcano_coords_"): #Skip if the file does not start with the matching string if there for some reason are other files in there (like the output if you're re-running this)
        continue
    comp = fn.replace("volcano_coords_", "").replace(".csv", "")
    df = pd.read_csv(os.path.join(coords_dir, fn)) #Loads the current volcano-coordinate .csv
    df["Chromosome"] = df["Gene"].map(gene_to_chr) #Assign each gene to a chromosome    
    n_unknown = int((df["Chromosome"] == "Other/Unknown").sum()) #Count any genes that couldn't be assigned
    if n_unknown:
        print(f"  {comp}: {n_unknown} genes could not be assigned to a chromosome")
    
    sig = df[df["significant"] == True] #Makes a smaller DataFrame containing only significant genes

    #Count how many genes were tested per chromosome
    tested = (
        df.groupby("Chromosome")["Gene"]
        .count()
        .reset_index(name="n_tested")
    )
    #Count significantly upregulated genes
    up = (
        sig[sig["logFC"] > 0]
        #Count how many of those upregulated genes occur on each chromosome
        .groupby("Chromosome")["Gene"]
        .count()
        .reset_index(name="n_up")
    )
    #Count significantly downregulated genes
    down = (
        sig[sig["logFC"] < 0]
        #Count how many of those downregulated genes occur on each chromosome
        .groupby("Chromosome")["Gene"]
        .count()
        .reset_index(name="n_down")
    )
    #Merge the tested, up, and down tables
    merged = (
        tested
        .merge(up, on="Chromosome", how="left")
        .merge(down, on="Chromosome", how="left")
    )
    merged[["n_up", "n_down"]] = merged[["n_up", "n_down"]].fillna(0).astype(int) #Replace any missing counts with zero
    merged["n_sig"] = merged["n_up"] + merged["n_down"] #Total significant genes per chromosome
    merged["frac_sig"] = merged["n_sig"] / merged["n_tested"] #Fraction of tested genes that are significant
    merged["comparison"] = comp #Adds the current comparison name to every row
    rows.append(merged)

#Stacks all the comparison tables vertically into one big table
out = pd.concat(rows, ignore_index=True)
out = out[["comparison", "Chromosome", "n_tested", "n_up", "n_down", "n_sig", "frac_sig"]]

#And make a wide-format table
wide = (
    out.pivot_table(
        index="comparison",
        columns="Chromosome",
        values=["n_up", "n_down"],
        fill_value=0,
        aggfunc="sum",
    )
)
wide.columns = [f"{stat}_{chrom}" for stat, chrom in wide.columns] #TFlatten the column names
wide = wide.reset_index()

#Save the long-format table
out.to_csv(out_csv, index=False)
#Save the wide-format table
wide.to_csv(os.path.join(coords_dir, "sig_DEGs_per_chromosome_WIDE.csv"), index=False)
print("Saved:", out_csv)
print("Saved:", os.path.join(coords_dir, "sig_DEGs_per_chromosome_WIDE.csv"))

Saved: plotting/volcano_plots/gene_coordinates/sig_DEGs_per_chromosome_all_comparisons.csv
Saved: plotting/volcano_plots/gene_coordinates/sig_DEGs_per_chromosome_WIDE.csv
